# Train Model

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from tqdm import tqdm  
from sklearn.metrics import accuracy_score
from Load_dataset import train_loader, val_loader  

# Ensure GPU is Used
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

#Check Dataset Before Training
print(f"Training Samples: {len(train_loader.dataset)}")
print(f"Validation Samples: {len(val_loader.dataset)}")
print(f"Class Mapping: {train_loader.dataset.class_to_idx}") 



Class Mapping: {'fake': 0, 'real': 1}
Train dataset: 140002 images
Validation dataset: 39428 images
Test dataset: 10905 images
Batch of images shape: torch.Size([32, 3, 224, 224])
Batch of labels: tensor([0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0,
        0, 0, 1, 1, 1, 1, 0, 0])
Using device: cuda
Training Samples: 140002
Validation Samples: 39428
Class Mapping: {'fake': 0, 'real': 1}


In [2]:
# Load Pretrained ResNeXt Model
model = models.resnext50_32x4d(weights=models.ResNeXt50_32X4D_Weights.IMAGENET1K_V1)
# Modify Fully Connected Layer for Binary Classification
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 512),
    nn.ReLU(),
    nn.BatchNorm1d(512),
    nn.Dropout(0.3),
    nn.Linear(512, 1)  # Output for binary classification (Real vs. Fake)
)

# Move model to GPU
model = model.to(device)

# Print Model Summary (Check if modifications are correct)
print(model)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
      (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(128, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1

In [3]:
# Define Loss Function & Optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Function (Added Checkpoint Saving After Each Epoch)
def train_model(model, train_loader, val_loader, num_epochs=5):
    """
    Trains the deepfake detection model.
    """
    best_val_loss = float("inf")  

    for epoch in range(1, num_epochs + 1):
        print(f"\nEpoch {epoch}/{num_epochs}")

        # 🔹 Training Phase
        model.train()
        train_loss = 0.0
        train_preds = []
        train_labels = []

        for images, labels in tqdm(train_loader, desc="Training"):
            images, labels = images.to(device), labels.to(device).float()

            optimizer.zero_grad()
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_preds.extend(torch.sigmoid(outputs).cpu().detach().numpy())  
            train_labels.extend(labels.cpu().numpy())

        train_accuracy = accuracy_score(train_labels, (torch.tensor(train_preds) > 0.5).int())
        print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_accuracy:.4f}")

        # 🔹 Validation Phase
        model.eval()
        val_loss = 0.0
        val_preds = []
        val_labels = []

        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validating"):
                images, labels = images.to(device), labels.to(device).float()
                outputs = model(images).squeeze()
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                val_preds.extend(torch.sigmoid(outputs).cpu().detach().numpy())
                val_labels.extend(labels.cpu().numpy())

        val_accuracy = accuracy_score(val_labels, (torch.tensor(val_preds) > 0.5).int())
        print(f"Validation Loss: {val_loss:.4f} | Validation Accuracy: {val_accuracy:.4f}")

        # Save Best Model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "model_b4_finetuned.pth")
            print("Best Model Saved!")


# Start Training
num_epochs = 10
train_model(model, train_loader, val_loader, num_epochs=num_epochs)

print("Model Training Completed and Saved!")


Epoch 1/10


Training: 100%|██████████| 4376/4376 [23:14<00:00,  3.14it/s]


Train Loss: 1715.1685 | Train Accuracy: 0.8189


Validating: 100%|██████████| 1233/1233 [04:01<00:00,  5.10it/s]


Validation Loss: 639.2392 | Validation Accuracy: 0.7653
Best Model Saved!

Epoch 2/10


...

Validation Loss: 758.1932 | Validation Accuracy: 0.7442

Epoch 10/10


Training: 100%|██████████| 4376/4376 [16:32<00:00,  4.41it/s]


Train Loss: 877.9924 | Train Accuracy: 0.9181


Validating: 100%|██████████| 1233/1233 [01:18<00:00, 15.79it/s]


Validation Loss: 396.2827 | Validation Accuracy: 0.8678
Best Model Saved!
Model Training Completed and Saved!


## Fine Tune Model

In [4]:
# Load the previously trained model
model.load_state_dict(torch.load("model_b4_finetuned.pth"))
model = model.to(device)

# Unfreeze the last block (layer4) for fine-tuning
for param in model.layer4.parameters():  # Unfreeze last ResNeXt block
    param.requires_grad = True

# Use a lower learning rate for fine-tuning
optimizer = optim.AdamW(model.parameters(), lr=0.00001) 

# Train the model again (Fine-Tuning Phase)
num_epochs = 10
train_model(model, train_loader, val_loader, num_epochs=num_epochs)

# Save Fine-Tuned Model
torch.save(model.state_dict(), "model_finetuned.pth")
print("Fine-tuned model saved successfully!")




Epoch 1/10


Training: 100%|██████████| 4376/4376 [16:35<00:00,  4.39it/s]


Train Loss: 772.0677 | Train Accuracy: 0.9280


Validating: 100%|██████████| 1233/1233 [01:23<00:00, 14.80it/s]


Validation Loss: 416.2893 | Validation Accuracy: 0.8777
Best Model Saved!

Epoch 2/10


...

Validation Loss: 391.0295 | Validation Accuracy: 0.8868

Epoch 10/10


Training: 100%|██████████| 4376/4376 [16:35<00:00,  4.40it/s]


Train Loss: 671.6978 | Train Accuracy: 0.9368


Validating: 100%|██████████| 1233/1233 [01:19<00:00, 15.59it/s]


Validation Loss: 359.0552 | Validation Accuracy: 0.8877
Fine-tuned model saved successfully!
